## 任务 BOLD｜分阶段入口

使用本地 JSON 配置，先清点和审查 run，再分开执行统计、比较与报告。默认不开启计算。已经审核的 A/B 配置应显式登记任务参考、预处理方法和 Bootstrap 设备。第一次使用可先运行[合成教程](Task_BOLD_Tutorial.ipynb)，公开影像接入见[数据指南](../docs/PUBLIC_DATA.md)。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'run_bold.py').is_file())
config = None  # 修改为实际配置绝对路径

def phase(name):
    completed = subprocess.run([sys.executable, '-B', str(repo / 'run_bold.py'), name,
                                '--config', str(Path(config).resolve())],
                               capture_output=True, encoding='utf-8', check=False,
                               env={**os.environ, 'PYTHONIOENCODING': 'utf-8', 'MPLBACKEND': 'Agg'})
    if completed.returncode:
        raise RuntimeError(completed.stdout or completed.stderr)
    return json.loads(completed.stdout)

### 1. 核对 run 和衍生文件

同名采集任务可不同。先看头部与缺项，再登记已经完成的衍生文件。

In [ ]:
if config:
    display(phase('inventory'))
    display(phase('preprocess'))

### 2. 审核时序、设计与配准

确认每条事件路径、实际采样时刻、设计自由度和已登记的 SDC/STC。需更正事件时保存独立版本，并重新确认对应 run 的时序。

In [ ]:
if config:
    audit = phase('audit')
    display(json.loads(Path(audit['output']).read_text(encoding='utf-8')))

In [ ]:
if config:
    new_events = None
    target_run = None
    operator = ''  # 填写审核者姓名
    reason = ''  # 说明更换依据，提交后需重新审核时序
    confirmed = False
    if new_events and target_run and confirmed:
        sys.path.insert(0, str(repo / 'bold'))
        from bold_events import stage_events
        display(stage_events(config, target_run, new_events,
                             operator=operator, reason=reason, confirmed=confirmed))

### 3. 审核通过后才拟合

主静态结果是 GLM＋时间导数，主动态结果是 DoubleScanner，FIR 用于敏感性。A/B 只在已选主核宽计算。

In [ ]:
fit = False
if config and fit:
    display(phase('glm'))

### 4. 从保存结果比较与生成 HTML

更改 β 比较无需重新拟合。B 图属于探索性结果。按同一位置、同一帧和同一色阶核对叠加。

In [ ]:
make_compare = False
make_report = False
if config and make_compare:
    display(phase('compare'))
if config and make_report:
    report = phase('report')
    print(Path(report['output']).resolve())  # 在浏览器打开该 HTML